In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Silver

In [0]:
log("📖 Reading from Silver orders ...")
df_silver_orders = spark.table(TBL_SILVER_ORDERS)
display(df_silver_orders.limit(10))

## 2. Generate Unique Dates from order_date and ship_date

In [0]:
df_order_dates = df_silver_orders.select(F.col("order_date").alias("date"))
df_ship_dates = df_silver_orders.select(F.col("ship_date").alias("date"))

df_all_dates = df_order_dates.union(df_ship_dates).distinct().dropna()
display(df_all_dates.limit(10))

## 3. Build dim_date with Calendar Attributes

In [0]:
df_dim_date = df_all_dates.select(
    F.date_format(F.col("date"), "yyyyMMdd").cast("int").alias("date_id"),
    F.col("date"),
    F.year(F.col("date")).alias("year"),
    F.quarter(F.col("date")).alias("quarter"),
    F.month(F.col("date")).alias("month"),
    F.date_format(F.col("date"), "MMMM").alias("month_name"),
    F.dayofmonth(F.col("date")).alias("day"),
    F.dayofweek(F.col("date")).alias("day_of_week"),
    F.date_format(F.col("date"), "EEEE").alias("day_name"),
    F.weekofyear(F.col("date")).alias("week_of_year"),
    F.when(F.dayofweek(F.col("date")).isin([1, 7]), True).otherwise(False).alias("is_weekend") 
).orderBy("date")

log(f"Total date records: {df_dim_date.count():,}")

display(df_dim_date.limit(10))

## 4. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_DIM_DATE):
    log(f"Upserting into {TBL_GOLD_DIM_DATE} ...")

    delta_table = DeltaTable.forName(spark, TBL_GOLD_DIM_DATE)
    (
        delta_table.alias("target")
        .merge(
            df_dim_date.alias("source"),
            "target.date_id = source.date_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_DATE} upserted")

else:
    log(f"Creating {TBL_GOLD_DIM_DATE}")
    (
        df_dim_date.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TBL_GOLD_DIM_DATE)
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_DATE} created")